In [1]:
import numpy as np
import matplotlib.pyplot as plt 
from PIL import Image 
import torch
import torch.nn as nn
import torch.optim as optim
from torchdiffeq import odeint

In [3]:
#　画像読み込み
img= Image.open("Images/Peppers.png").convert("L")

#　解像度
img=img.resize((256,256))

# numpy化
img_array=np.array(img).astype(np.float32)

# 画素値（0～255）を0～1へ変換
img_array=img_array/255.0

In [4]:
#　画像の縦横が何マスか
height,width=img_array.shape

x=np.linspace(1,-1,width)
y=np.linspace(1,-1,height)

X,Y=np.meshgrid(x,y)

In [5]:
print(X)

[[ 1.          0.99215686  0.98431373 ... -0.98431373 -0.99215686
  -1.        ]
 [ 1.          0.99215686  0.98431373 ... -0.98431373 -0.99215686
  -1.        ]
 [ 1.          0.99215686  0.98431373 ... -0.98431373 -0.99215686
  -1.        ]
 ...
 [ 1.          0.99215686  0.98431373 ... -0.98431373 -0.99215686
  -1.        ]
 [ 1.          0.99215686  0.98431373 ... -0.98431373 -0.99215686
  -1.        ]
 [ 1.          0.99215686  0.98431373 ... -0.98431373 -0.99215686
  -1.        ]]


In [6]:
print(Y)

[[ 1.          1.          1.         ...  1.          1.
   1.        ]
 [ 0.99215686  0.99215686  0.99215686 ...  0.99215686  0.99215686
   0.99215686]
 [ 0.98431373  0.98431373  0.98431373 ...  0.98431373  0.98431373
   0.98431373]
 ...
 [-0.98431373 -0.98431373 -0.98431373 ... -0.98431373 -0.98431373
  -0.98431373]
 [-0.99215686 -0.99215686 -0.99215686 ... -0.99215686 -0.99215686
  -0.99215686]
 [-1.         -1.         -1.         ... -1.         -1.
  -1.        ]]


In [ ]:
# 欠損作成
mask=np.ones(img_array.shape,dtype=bool)


np.random.seed(42)
num_missing=int(height*width*0.05)

#　↓この条件をもとに欠損させる場所を指定
indices= np.random.choice(
    height*width,
    num_missing,
    replace=False #「重複無し」の意味。一度選んだピクセルの位置は二度とえらばない→正確にかぶりなく5％分の場所を確保できる
)

# リスト内包表記
# 5%分のランダムな通し番号（indices）から、数字を1つずつ順番に取り出して idx（インデックス）という変数に入れています。この処理をデータの数だけぐるぐる繰り返します。
# idx // width（スラッシュ2つ）：idx を画像の横幅（width）で割った「商（割り算の答えの整数部分）」です。これが「縦（上から何行目か）」を表します。
# idx % width（パーセント）：idx を画像の横幅（width）で割った「余り」です。これが「横（左から何列目か）」を表します。
"""
例えば、横幅（width）が 100 の画像があるとします。
通し番号（idx）が 253 番だった場合、これを縦と横に直すとどうなるでしょうか？
・縦： 253 // 100 = 2 （上から3行目 ※0から数えるため）
・横： 253 % 100 = 53 （左から54列目）
つまり、253番という1列の背番号が、(2, 53) という「2行目の53列目」という画像の具体的なマス目の位置（座標）に見事変換されます。
"""
missing_points=[
    (idx//width,idx % width)
    for idx in indices
]